## Ayudantía 2
### Profesor: Elwin van 't Wout
### Ayudante: Alberto Almuna Morales (alberto.almuna@uc.cl)

Se propone el siguiente ejemplo a paralelizar:

In [8]:
import time as tm
import numpy as np

In [9]:
def calculateHaversineDistance(latitude1, longitude1, latitude2, longitude2):
    earth_radius_km = 6371

    latitude_dist = (latitude1 - latitude2) * np.pi / 180
    longitude_dist = (longitude1 - longitude2) * np.pi / 180

    haversine_value = np.sin(latitude_dist / 2)**2 + \
        np.sin(longitude_dist / 2)**2
    haversine_value *= np.cos(latitude1 * np.pi / 180) * \
        np.cos(latitude2 * np.pi / 180)

    haversine_dist = 2 * \
        np.arcsin(np.sqrt(haversine_value)) * earth_radius_km

    return haversine_dist

Código secuencial:

In [10]:
t0 = tm.time()

coordinates = []
max_distance = -1

with open("coordenadas10000.txt", 'r') as file:

    file.readline()

    for line in file:

        coordinates_info = line.strip().split(',')
        latitude = float(coordinates_info[0])
        longitude = float(coordinates_info[1])

        coordinates.append((latitude, longitude))


for coordinate1 in coordinates:
    for coordinate2 in coordinates:

        distance = calculateHaversineDistance(
            coordinate1[0], coordinate1[1], coordinate2[0], coordinate2[1])

        if distance > max_distance:
            max_distance = distance


t1 = tm.time()


print('Tiempo de ejecucion: ', t1-t0)
print('Distancia maxima: ', max_distance)

Tiempo de ejecucion:  589.8582360744476
Distancia maxima:  44.829257919253564


Código paralelo:

In [ ]:
import time as tm
import numpy as np
from joblib import Parallel
from joblib import delayed

In [11]:
def readData(chunk):
    coordinates = []
    
    for line in chunk:
        coordinates_info = line.strip().split(',')
        latitude = float(coordinates_info[0])
        longitude = float(coordinates_info[1])

        coordinates.append((latitude, longitude))
    return coordinates

In [12]:
def calculateDistances(chunk, original):
    max_distance = -1
    for coordinate1 in chunk:
        for coordinate2 in original:

            distance = calculateHaversineDistance(
                coordinate1[0], coordinate1[1], coordinate2[0], coordinate2[1])

            if distance > max_distance:
                max_distance = distance
    return max_distance

In [13]:
t0 = tm.time()
workers = 8

with open("coordenadas10000.txt", 'r') as file:
    # Leemos el archivo:
    file.readline() # Nos saltamos la primera línea
    file = file.readlines()

    # Separamos la información en chunks de datos:
    chunks_list = []
    if workers > 1:
        chunk_size = int(len(file)/workers)
        for i in range(workers-1):
            chunks_list.append(file[i*chunk_size:(i+1)*chunk_size])
        chunks_list.append(file[(i+1)*chunk_size:])
    else:
        chunks_list = [file]

    # Creamos y ejecutamos los procesos paralelos:

    # Primero hacemos la lectura de la información:
    parallel_pool = Parallel(n_jobs=workers)
    parallel_readData = delayed(readData)
    parallel_tasks1 = map(parallel_readData, chunks_list)
    parallel_results1 = parallel_pool(parallel_tasks1)

    # Creamos una lista con todas las coordenadas:
    original = np.concatenate(parallel_results1)

    # Luego los cálculos de distancias
    parallel_calculateDistances = delayed(calculateDistances)
    parallel_tasks2 = [parallel_calculateDistances(i, original) for i in parallel_results1]
    parallel_results2 = parallel_pool(parallel_tasks2)
    max_distance = max(parallel_results2)

t1 = tm.time()

# Imprimimos las estadísticas:
print('Tiempo de ejecucion: ', t1-t0)
print('Distancia maxima: ', max_distance)

Tiempo de ejecucion:  169.1912407875061
Distancia maxima:  44.829257919253564
